# Task A -- more capacity, or more epochs?

Two knobs that have never been touched on either task, run as one session because both ask
the same question: is the current recipe underfitting?

## The evidence that says it might be

I read the per-epoch validation curve out of the existing Task A logs. Averaged over 14
fold-runs, macro-F1 is **still rising when training stops**:

| epoch | mean val macro-F1 | gain |
|---|---|---|
| 3 | 0.7745 | |
| 4 | 0.7852 | +0.0107 |
| 5 | 0.7906 | +0.0054 |
| 6 | 0.7944 | +0.0038 |

Six epochs is a cutoff, not a converged point. Task B showed the same shape, with 13 of 20
folds peaking at the final epoch. The gains are halving each epoch though, so extrapolation
suggests epochs 7 to 10 are worth only a few tenths of a point.

Capacity is the other reading of the same symptom. MuRIL-large has 24 layers against 12,
and Task A's 6,401 rows support a larger model far better than Task B's 3,143. It was cut
by a budget guard in Task B's first sweep and never retried anywhere.

| arm | change | risk |
|---|---|---|
| `task_a_large` | `google/muril-large-cased`, `--no-fgm`, lower learning rate, smaller micro-batch | high: on Task B, mDeBERTa collapsed to 0.1002 because the shared learning rate was too high for it |
| `task_a_ep10` | `--epochs 10` on the base model | low: the curve above says it cannot be much worse |

## Settings that differ for the large model, and why

`--no-fgm`: FGM clones the embedding table every step. With a 24-layer model and a T4's
16GB that is what makes it not fit.

`--lr 1e-5` instead of 3e-5: larger encoders need smaller steps, and 3e-5 is precisely
what collapsed mDeBERTa on Task B.

`--bs 4 --grad-accum 4`: same effective batch of 16, a quarter of the activations.

## Runtime

Roughly **8 hours** for both, and the large arm is the uncertain one, so it runs first. A
smoke pass on a single fold checks it has not collapsed before the remaining folds are
spent on it.

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Smoke-check MuRIL-large on one fold

A collapsed large model produces a degenerate classifier that predicts one class, and that
is visible in a single fold. Spending 480 minutes to discover it would be waste, so one
fold is run first and the remaining arms are skipped if it has collapsed.

The check is on the predicted class balance, not on the score: Task A is 49% Hate, so any
arm calling one class on over 90% of rows has failed regardless of its macro-F1.

In [ ]:
import time
t0 = time.time()
BUDGET_H, RESERVE_MIN = 10.5, 20
left = lambda: BUDGET_H * 3600 - (time.time() - t0) - RESERVE_MIN * 60

from sklearn.metrics import f1_score
from hastika.common.preprocessing import clean
df = train.iloc[keep].reset_index(drop=True)
X = np.array([clean(t, demojize=True) for t in df["Comment"]])
y = (df["Label"] == "Hate").astype(int).values
oof_of = lambda tag: np.load(pathlib.Path("artifacts/runs") / tag / "oof_probs.npy")
score_of = lambda tag: f1_score(y, oof_of(tag).argmax(1), average="macro")

LARGE = ["--model", "google/muril-large-cased", "--no-fgm", "--lr", "1e-5",
         "--bs", "4", "--grad-accum", "4", "--eval-bs", "16"]
SMOKE = "task_a_large_smoke"
large_ok = False
if left() > 100 * 60:
    run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", SMOKE,
         *LARGE, "--folds", "0", "--epochs", "6", "--select", "last",
         "--reinit-layers", "1", "--seeds", "42"], log=f"artifacts/logs/{SMOKE}.log")
    hp = np.load(pathlib.Path("artifacts/runs") / SMOKE / "holdout_probs.npy")
    mask = hp.sum(1) > 0
    rate = hp[mask].argmax(1).mean()
    print(f"\nholdout rows scored: {int(mask.sum())}; predicted Hate rate {rate:.3f} "
          f"(Task A prior 0.491)")
    large_ok = 0.10 < rate < 0.90
    print("large model looks healthy" if large_ok else "LARGE MODEL COLLAPSED -- skipping it")
else:
    print("no budget for the large smoke test")

## 2. The arms

Five folds each, on the same split seed as every other Task A experiment so the results
are comparable with Runs 11, 12 and 13.

In [ ]:
# --eval-bs lives in the per-arm lists, never here: the large arm needs a smaller one
# and a duplicated flag in one command line is a silent trap to leave lying around.
COMMON = ["--folds", "5", "--select", "last", "--reinit-layers", "1", "--seeds", "42"]
BASE = ["--model", "google/muril-base-cased", "--bs", "8", "--grad-accum", "2",
        "--eval-bs", "32"]
ARMS = []
if large_ok:
    ARMS.append(("task_a_large_5f", [*LARGE, "--epochs", "6"], 480))
ARMS += [("task_a_ep10_5f", [*BASE, "--epochs", "10"], 265),
         ("task_a_ep6_5f",  [*BASE, "--epochs", "6"],  160)]

ran = []
for tag, extra, est in ARMS:
    if left() < est * 60:
        print(f"skip {tag}: {left()/60:.0f} min left, needs ~{est}", flush=True)
        continue
    run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", tag,
         *COMMON, *extra], log=f"artifacts/logs/{tag}.log")
    ran.append(tag)
print("\narms completed:", ran)

## 3. Compare

`task_a_ep6_5f` is the six-epoch control and runs last, so a short session still has the
ten-epoch and large numbers. If it gets skipped, compare against Run 11's one-layer
out-of-fold score instead, which uses identical settings.

In [ ]:
from sklearn.metrics import classification_report
for tag in ran:
    print(f"  {tag:20s} OOF macro-F1 {score_of(tag):.4f}")
if "task_a_ep6_5f" in ran:
    base = score_of("task_a_ep6_5f")
    for tag in ran:
        if tag != "task_a_ep6_5f":
            print(f"    {tag:20s} {score_of(tag) - base:+.4f} vs the six-epoch control")
print("\n  fold noise on 6,401 rows is about 0.006")
for tag in ran:
    print(f"\n=== {tag} ===")
    print(classification_report(y, oof_of(tag).argmax(1),
                                target_names=["Non-Hate", "Hate"], digits=3))

## 4. Read the epoch curve directly

If the ten-epoch arm ran, its own per-epoch validation trace says whether ten is enough or
whether the curve is still climbing at ten as well.

In [ ]:
import re, collections
for tag in ran:
    vals = collections.defaultdict(list)
    for line in pathlib.Path(f"artifacts/logs/{tag}.log").read_text().splitlines():
        m = re.search(r"ep(\d+) \d+/\d+ loss [\d.]+ val ([\d.]+)", line)
        if m:
            vals[int(m.group(1))].append(float(m.group(2)))
    if not vals:
        continue
    print(f"\n{tag}: mean validation macro-F1 by epoch")
    prev = None
    for ep in sorted(vals):
        v = float(np.mean(vals[ep]))
        print(f"  epoch {ep:2d}  {v:.4f}" + ("" if prev is None else f"   {v - prev:+.4f}"))
        prev = v

## 5. Preserve outputs

The large checkpoint is several gigabytes and is not worth downloading unless it won.
Take the `oof_probs.npy` files and the logs.

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_capacity_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for tag in ran + [SMOKE]:
    for name in ["oof_probs.npy", "holdout_probs.npy", "test_probs.npy"]:
        p = pathlib.Path("artifacts/runs") / tag / name
        if p.exists():
            shutil.copy2(p, OUT / f"{tag}_{name}")
    log = pathlib.Path(f"artifacts/logs/{tag}.log")
    if log.exists():
        shutil.copy2(log, OUT / log.name)
json.dump({t: score_of(t) for t in ran}, open(OUT / "oof_scores.json", "w"), indent=2)
print(sorted(x.name for x in OUT.iterdir()))

## 6. What to do with the result

Record every arm, including a collapsed large model if that is what happened. A collapse
is a useful result: it is what stops the same idea being retried in three weeks.

If ten epochs wins by more than a point, the epoch count changes for every later Task A
run. If MuRIL-large wins, it becomes a candidate encoder, but check its per-class report
before adopting it, because a large model on 6,401 rows can gain overall while getting
worse on one class.

No submission is produced here. Both arms are decisions, not builds.